# THỰC HÀNH HADOOP TRÊN GOOGLE COLAB
**Môn:** Big Data & Cloud Computing  
**Chủ đề:** HDFS, Block, Replication, YARN và MapReduce  
**Môi trường:** Google Colab + Apache Hadoop 3.5.0 + Java 17

> Mục tiêu của notebook là để sinh viên thực hành Hadoop ngay trên trình duyệt, không phải cài Hadoop trên máy cá nhân.

## Sau bài thực hành, sinh viên có thể
1. Phân biệt **Local File System** và **HDFS**.
2. Thực hiện các lệnh HDFS cơ bản: `mkdir`, `put`, `ls`, `cat`, `cp`, `mv`, `rm`, `get`.
3. Quan sát file HDFS được chia thành **block**.
4. Giải thích ý nghĩa của **replication** và giới hạn của mô hình single-node trên Colab.
5. Khởi động và kiểm tra **YARN**.
6. Chạy một chương trình **MapReduce WordCount**.
7. Vận dụng MapReduce để tổng hợp dữ liệu bán hàng và kiểm chứng kết quả.

---
### Lưu ý quan trọng
Google Colab chạy một máy ảo tạm thời. Vì vậy notebook này sử dụng Hadoop ở chế độ **pseudo-distributed single-node**:
- NameNode, DataNode, ResourceManager và NodeManager chạy thành các tiến trình riêng.
- Có thể thực hành HDFS, YARN và MapReduce.
- Không thể mô phỏng đầy đủ fault tolerance của cụm nhiều DataNode như hệ thống production.

## PHẦN 0 — Khởi tạo môi trường Hadoop

Cell dưới đây:
- kiểm tra Java 17 (Hadoop 3.5 yêu cầu Java 17 ở phía server),
- tải Apache Hadoop 3.5.0,
- cấu hình HDFS và YARN,
- khởi động SSH,
- format NameNode lần đầu,
- khởi động HDFS và YARN.

> Chạy cell này trước. Những lần Runtime bị reset, hãy chạy lại từ đầu.

In [ ]:
%%bash
set -e

HADOOP_VERSION=3.5.0
HADOOP_HOME=/content/hadoop-${HADOOP_VERSION}

echo "=== 1. Java ==="
java -version

echo "=== 2. Install SSH if needed ==="
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openssh-server rsync > /dev/null

echo "=== 3. Download Hadoop ${HADOOP_VERSION} ==="
if [ ! -d "$HADOOP_HOME" ]; then
  FILE=hadoop-${HADOOP_VERSION}.tar.gz
  URL1=https://downloads.apache.org/hadoop/common/hadoop-${HADOOP_VERSION}/${FILE}
  URL2=https://archive.apache.org/dist/hadoop/common/hadoop-${HADOOP_VERSION}/${FILE}
  wget -q "$URL1" -O /content/$FILE || wget -q "$URL2" -O /content/$FILE
  tar -xzf /content/$FILE -C /content
fi

echo "=== 4. JAVA_HOME ==="
JAVA_HOME=$(dirname "$(dirname "$(readlink -f "$(command -v java)")")")
echo "JAVA_HOME=$JAVA_HOME"

echo "=== 5. Configure passwordless SSH ==="
mkdir -p ~/.ssh
chmod 700 ~/.ssh
if [ ! -f ~/.ssh/id_rsa ]; then
  ssh-keygen -q -t rsa -N "" -f ~/.ssh/id_rsa
fi
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
sort -u ~/.ssh/authorized_keys -o ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys
service ssh restart > /dev/null
ssh-keyscan -H localhost >> ~/.ssh/known_hosts 2>/dev/null || true

CONF=$HADOOP_HOME/etc/hadoop

cat > "$CONF/core-site.xml" <<'EOF'
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
</configuration>
EOF

cat > "$CONF/hdfs-site.xml" <<'EOF'
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:/content/hadoop_data/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:/content/hadoop_data/datanode</value>
  </property>
</configuration>
EOF

cat > "$CONF/mapred-site.xml" <<'EOF'
<configuration>
  <property>
    <name>mapreduce.framework.name</name>
    <value>yarn</value>
  </property>
  <property>
    <name>mapreduce.application.classpath</name>
    <value>$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/*:$HADOOP_MAPRED_HOME/share/hadoop/mapreduce/lib/*</value>
  </property>
</configuration>
EOF

cat > "$CONF/yarn-site.xml" <<'EOF'
<configuration>
  <property>
    <name>yarn.nodemanager.aux-services</name>
    <value>mapreduce_shuffle</value>
  </property>
  <property>
    <name>yarn.nodemanager.env-whitelist</name>
    <value>JAVA_HOME,HADOOP_COMMON_HOME,HADOOP_HDFS_HOME,HADOOP_CONF_DIR,CLASSPATH_PREPEND_DISTCACHE,HADOOP_YARN_HOME,HADOOP_HOME,PATH,LANG,TZ,HADOOP_MAPRED_HOME</value>
  </property>
  <property>
    <name>yarn.nodemanager.resource.memory-mb</name>
    <value>4096</value>
  </property>
  <property>
    <name>yarn.scheduler.maximum-allocation-mb</name>
    <value>4096</value>
  </property>
</configuration>
EOF

if ! grep -q "^export JAVA_HOME=" "$CONF/hadoop-env.sh"; then
  echo "export JAVA_HOME=$JAVA_HOME" >> "$CONF/hadoop-env.sh"
fi

echo "=== 6. Stop old Hadoop daemons if any ==="
$HADOOP_HOME/sbin/stop-yarn.sh >/dev/null 2>&1 || true
$HADOOP_HOME/sbin/stop-dfs.sh  >/dev/null 2>&1 || true

echo "=== 7. Format HDFS only if not formatted ==="
mkdir -p /content/hadoop_data
if [ ! -d /content/hadoop_data/namenode/current ]; then
  $HADOOP_HOME/bin/hdfs namenode -format -force -nonInteractive > /dev/null
fi

echo "=== 8. Start HDFS and YARN ==="
$HADOOP_HOME/sbin/start-dfs.sh
$HADOOP_HOME/sbin/start-yarn.sh

echo
echo "=== 9. Running Java processes ==="
jps

echo
echo "Hadoop is ready."


### Kiểm tra kết quả

Kết quả `jps` kỳ vọng có các tiến trình tương tự:

```text
NameNode
DataNode
SecondaryNameNode
ResourceManager
NodeManager
```

**Câu hỏi 0.1**
- Tiến trình nào quản lý metadata của HDFS?
- Tiến trình nào lưu dữ liệu?
- Tiến trình nào quản lý tài nguyên của YARN?

In [ ]:
import os
os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
os.environ["HADOOP_CONF_DIR"] = "/content/hadoop-3.5.0/etc/hadoop"
os.environ["JAVA_HOME"] = os.path.dirname(os.path.dirname(os.path.realpath("/usr/bin/java")))
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/sbin:" + os.environ["PATH"]

print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
!hadoop version | head -n 5
!jps

# BÀI 1 — Local File System và HDFS

Trong bài giảng, HDFS là nền tảng lưu trữ của Hadoop.

```text
Local file
/content/sample.txt

        hdfs dfs -put

HDFS file
/user/student/input/sample.txt
```

## 1.1 Tạo dữ liệu trên máy local

In [ ]:
sample_lines = [
    "Hadoop stores big data",
    "Hadoop uses HDFS",
    "HDFS stores data in blocks",
    "YARN manages resources",
    "MapReduce processes data",
    "Hadoop supports distributed processing",
    "HDFS supports scalable storage",
]
with open("/content/sample.txt", "w") as f:
    f.write("\n".join(sample_lines) + "\n")

!ls -lh /content/sample.txt
!cat /content/sample.txt

## 1.2 Tạo thư mục trên HDFS và upload file

In [ ]:
!hdfs dfs -mkdir -p /user/student/input
!hdfs dfs -put -f /content/sample.txt /user/student/input/
!hdfs dfs -ls -h /user/student/input

## 1.3 Đọc dữ liệu từ HDFS

In [ ]:
!hdfs dfs -cat /user/student/input/sample.txt

## 1.4 Thực hành các lệnh HDFS cơ bản

In [ ]:
!hdfs dfs -cp -f /user/student/input/sample.txt /user/student/input/sample_copy.txt
!hdfs dfs -mv /user/student/input/sample_copy.txt /user/student/input/sample_backup.txt
!hdfs dfs -ls -h /user/student/input
!hdfs dfs -du -h /user/student/input

## 1.5 Download từ HDFS về local

In [ ]:
!rm -f /content/downloaded_sample.txt
!hdfs dfs -get /user/student/input/sample.txt /content/downloaded_sample.txt
!ls -lh /content/downloaded_sample.txt
!cat /content/downloaded_sample.txt

### Bài tập 1
Sinh viên tự thực hiện:

1. Tạo thư mục `/user/student/lab01`.
2. Tạo file local `student.txt` chứa MSSV, họ tên, lớp.
3. Upload file vào HDFS.
4. Kiểm tra bằng `hdfs dfs -ls`.
5. Đọc nội dung bằng `hdfs dfs -cat`.
6. Download file từ HDFS về `/content/result_student.txt`.

**Minh chứng:** chụp kết quả bước 4 và 5.

In [ ]:
# TODO - BÀI TẬP 1

# BÀI 2 — Block trong HDFS

HDFS chia file thành các **block**.

```text
File
 ├── Block 1
 ├── Block 2
 └── Block 3
```

## 2.1 Kiểm tra block size mặc định

In [ ]:
!hdfs getconf -confKey dfs.blocksize

## 2.2 Tạo file khoảng 260 MB để quan sát nhiều block

In [ ]:
!dd if=/dev/zero of=/content/bigfile.bin bs=1M count=260 status=none
!ls -lh /content/bigfile.bin
!hdfs dfs -mkdir -p /user/student/blocks
!hdfs dfs -put -f /content/bigfile.bin /user/student/blocks/

## 2.3 Quan sát block bằng fsck

In [ ]:
!hdfs fsck /user/student/blocks/bigfile.bin -files -blocks -locations

### Câu hỏi 2
1. File có bao nhiêu block?
2. Block cuối có kích thước bằng block size mặc định không?
3. Các block đang nằm trên bao nhiêu DataNode?
4. Vì sao trên Colab chỉ thấy một DataNode?

# BÀI 3 — Replication và Fault Tolerance

Trong cluster thật, HDFS có thể tạo nhiều bản sao của block:

```text
Block A
 ├── DataNode 1
 ├── DataNode 2
 └── DataNode 3
```

Google Colab chỉ có **một DataNode**, vì vậy không thể mô phỏng đầy đủ fault tolerance của cụm nhiều node.

In [ ]:
!hdfs dfs -stat "Replication = %r | Size = %b bytes | File = %n" /user/student/blocks/bigfile.bin
!hdfs dfsadmin -report

## 3.1 Yêu cầu replication = 3 và quan sát under-replication

In [ ]:
!hdfs dfs -setrep 3 /user/student/blocks/bigfile.bin
!hdfs fsck /user/student/blocks/bigfile.bin -files -blocks -locations | tail -n 30

### Câu hỏi 3
1. `replication factor = 3` nghĩa là gì trong cluster thật?
2. Vì sao file có thể bị báo **under-replicated** trên Colab?
3. Nếu cluster thật có 3 DataNode và replication factor = 3, điều gì xảy ra khi một DataNode hỏng?

In [ ]:
!hdfs dfs -setrep -w 1 /user/student/blocks/bigfile.bin

# BÀI 4 — HDFS Administration

## 4.1 Trạng thái HDFS

In [ ]:
!hdfs dfsadmin -report

## 4.2 Kiểm tra filesystem

In [ ]:
!hdfs fsck /

### Bài tập 4
Giải thích:
- Live datanodes
- DFS Used
- DFS Remaining
- Under replicated blocks
- Missing blocks

# BÀI 5 — YARN

Các thành phần chính:

- **ResourceManager:** điều phối tài nguyên toàn cụm.
- **NodeManager:** quản lý container trên từng worker node.
- **ApplicationMaster:** điều phối một application attempt.
- **Container:** phần tài nguyên được cấp để chạy một tiến trình/tác vụ.

```text
Client → ResourceManager → container đầu tiên chạy ApplicationMaster
                              ↓
ApplicationMaster → ResourceManager (xin container cho tác vụ)
                              ↓
ApplicationMaster → NodeManager → task containers
```

> Container không phải máy ảo hay Docker container; trong YARN, nó là một mức cấp phát tài nguyên (ví dụ memory và vcores) trên một node.


## 5.1 Kiểm tra NodeManager

In [ ]:
!yarn node -list -all

## 5.2 Kiểm tra application

In [ ]:
!yarn application -list -appStates ALL

### Câu hỏi 5
1. Có bao nhiêu NodeManager trong Colab?
2. ResourceManager và NodeManager khác nhau thế nào?
3. ApplicationMaster thuộc toàn cluster hay từng application attempt?
4. Container trong YARN biểu diễn điều gì?
5. Sau khi client submit application, container nào được cấp trước và dùng để làm gì?


# BÀI 6 — MapReduce WordCount

```text
Input
  ↓
Map
  ↓
Shuffle / Sort
  ↓
Reduce
  ↓
Output
```

## 6.1 Xóa output cũ

In [ ]:
!hdfs dfs -rm -r -f /user/student/output_wordcount

## 6.2 Chạy WordCount trên YARN

In [ ]:
import glob
EXAMPLES = glob.glob("/content/hadoop-3.5.0/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar")[0]
print(EXAMPLES)
!hadoop jar $EXAMPLES wordcount /user/student/input/sample.txt /user/student/output_wordcount

## 6.3 Xem kết quả

In [ ]:
!hdfs dfs -ls /user/student/output_wordcount
!hdfs dfs -cat /user/student/output_wordcount/part-r-00000

## 6.4 Xem application trên YARN

In [ ]:
!yarn application -list -appStates ALL

### Câu hỏi 6
1. Từ nào xuất hiện nhiều nhất?
2. Output được lưu ở local hay HDFS?
3. Vì sao Hadoop không cho chạy job nếu thư mục output đã tồn tại?
4. Job WordCount được YARN quản lý như thế nào?

# BÀI 7 — Bài toán bán hàng

Dataset:

```text
order_id,city,product,amount
```

Mục tiêu: **tính tổng doanh thu theo thành phố**.

In [ ]:
sales_lines = [
    "order_id,city,product,amount",
    "1,HCMC,Laptop,1200",
    "2,Hanoi,Phone,800",
    "3,HCMC,Phone,900",
    "4,Danang,Tablet,600",
    "5,Hanoi,Laptop,1300",
    "6,HCMC,Tablet,650",
    "7,Danang,Phone,750",
    "8,Hanoi,Tablet,700",
    "9,HCMC,Laptop,1250",
    "10,Danang,Laptop,1200",
]
with open("/content/sales.csv", "w") as f:
    f.write("\n".join(sales_lines) + "\n")
!cat /content/sales.csv

## 7.1 Upload dataset lên HDFS

In [ ]:
!hdfs dfs -mkdir -p /user/student/sales
!hdfs dfs -put -f /content/sales.csv /user/student/sales/
!hdfs dfs -ls -h /user/student/sales

## 7.2 Thiết kế MapReduce

Mapper:

```text
1,HCMC,Laptop,1200
        ↓
HCMC    1200
```

Reducer:

```text
HCMC [1200,900,650,1250]
        ↓
HCMC 4000
```

## 7.3 Viết Python Mapper và Reducer

In [ ]:
mapper_code = """#!/usr/bin/env python3
import sys
for line in sys.stdin:
    line = line.strip()
    if not line or line.startswith("order_id"):
        continue
    parts = line.split(",")
    city = parts[1]
    amount = float(parts[3])
    print(f"{city}\\t{amount}")
"""

reducer_code = """#!/usr/bin/env python3
import sys
current_city = None
total = 0.0

for line in sys.stdin:
    city, amount = line.strip().split("\\t")
    amount = float(amount)
    if current_city == city:
        total += amount
    else:
        if current_city is not None:
            print(f"{current_city}\\t{total:.0f}")
        current_city = city
        total = amount

if current_city is not None:
    print(f"{current_city}\\t{total:.0f}")
"""

open("/content/mapper.py", "w").write(mapper_code)
open("/content/reducer.py", "w").write(reducer_code)
!chmod +x /content/mapper.py /content/reducer.py

### Chạy thử Mapper riêng

In [ ]:
!cat /content/sales.csv | python3 /content/mapper.py

### Chạy Hadoop Streaming

In [ ]:
!hdfs dfs -rm -r -f /user/student/sales_output

import glob
STREAMING = glob.glob("/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming-*.jar")[0]
print(STREAMING)

!hadoop jar $STREAMING \
  -files /content/mapper.py,/content/reducer.py \
  -mapper mapper.py \
  -reducer reducer.py \
  -input /user/student/sales/sales.csv \
  -output /user/student/sales_output

### Xem kết quả

In [ ]:
!hdfs dfs -cat /user/student/sales_output/part-00000

### Bài tập 7
Chọn **một** trong ba yêu cầu, sửa Mapper/Reducer và đối chiếu kết quả bằng Python hoặc tính tay:

- **7A.** Tổng doanh thu theo sản phẩm.
- **7B.** Số đơn hàng theo thành phố.
- **7C.** Doanh thu trung bình theo thành phố (Reducer phải duy trì cả tổng và số lượng).

**Minh chứng cần nộp:** mã đã sửa, lệnh chạy job, output HDFS và bảng kết quả kỳ vọng.


In [ ]:
# TODO - BÀI TẬP 7

# BÀI 8 — Bài tổng hợp

Cho dataset:

```text
transaction_id,customer,category,amount
1,C01,Food,150
2,C02,Book,300
3,C01,Book,200
4,C03,Food,220
5,C02,Electronics,1500
6,C01,Electronics,1000
7,C03,Book,250
8,C02,Food,180
```

Thực hiện:

1. Tạo file local và upload lên `/user/student/final/input`.
2. Kiểm tra file trên HDFS bằng `ls`, `cat` và `stat`.
3. Viết Hadoop Streaming để tính **tổng chi tiêu theo category**.
4. Lưu output vào `/user/student/final/output` và hiển thị kết quả.
5. Dùng Python thuần hoặc tính tay để tạo kết quả kỳ vọng, rồi đối chiếu với MapReduce.
6. Xem trạng thái application bằng YARN và giải thích Map → Shuffle/Sort → Reduce trong lời giải.

**Yêu cầu tái chạy:** xóa output cũ trước khi chạy lại job.


In [ ]:
# TODO - BÀI TẬP 8

# BÀI 9 — Câu hỏi củng cố

1. HDFS giải quyết bài toán lưu trữ nào và không phù hợp với workload nào?
2. NameNode lưu metadata gì? DataNode lưu gì?
3. Vì sao HDFS chia file thành block?
4. Replication tăng khả năng chịu lỗi và thông lượng đọc như thế nào?
5. Vì sao replication factor = 3 không thể đạt đủ trên Colab single-node?
6. SecondaryNameNode có phải NameNode dự phòng nóng không? Giải thích.
7. ResourceManager, NodeManager và ApplicationMaster khác nhau thế nào?
8. Container trong YARN là gì?
9. Mô tả đúng thứ tự từ lúc client submit application đến khi task chạy.
10. Mô tả Map → Shuffle/Sort → Reduce và vị trí lưu output.


# BÀI 10 — Thu bài

## Sinh viên nộp
- Notebook `.ipynb` đã chạy có output.
- Minh chứng:
  1. `jps`
  2. `hdfs dfs -ls`
  3. `hdfs fsck ... -blocks -locations`
  4. `hdfs dfsadmin -report`
  5. `yarn node -list -all`
  6. WordCount output
  7. Sales MapReduce output
  8. Bài tổng hợp cuối

## Rubric / 10 điểm

| Nội dung | Điểm |
|---|---:|
| Khởi động Hadoop và architecture | 1.0 |
| HDFS cơ bản | 1.5 |
| Block và replication | 1.5 |
| HDFS administration | 1.0 |
| YARN | 1.0 |
| WordCount | 1.5 |
| Sales MapReduce | 1.5 |
| Bài tổng hợp | 1.0 |

# MỞ RỘNG — Sinh viên khá/giỏi

1. Tăng dataset lên 100.000 hoặc 1.000.000 dòng.
2. So sánh thời gian local Python với Hadoop MapReduce.
3. Thêm Combiner.
4. Thử 2 reducer:
   ```bash
   -D mapreduce.job.reduces=2
   ```
5. Giải thích khi nào Hadoop MapReduce không phù hợp.